# Phase 2: the `ft` and `combined` arms

This is section 6.2's second and third deliveries. `arm_c` already
covers the first.

| arm | phase-1 adapter | this instance's evidence in the weights | evidence in the prompt |
| --- | --- | --- | --- |
| `none` | no | no | no |
| `icl` | no | no | yes |
| `arm_c` | yes | no | yes |
| **`ft`** | yes | **yes** | no |
| **`combined`** | yes | **yes** | yes |

`ft` asks whether an observation log is usable at all once it has been
absorbed rather than read. `combined` asks whether the two routes add or
interfere. Together with `icl` and `arm_c` that is the full 2x2 over
where the evidence lives.

## How it works

Phase 2 is per instance. Each payload gets its own adapter: the phase-1
weights, then `PHASE2_STEPS` more steps on that payload's own evidence
alone, with the loss on the evidence tokens. Then the four probes are
asked twice, once with an empty context and once with the evidence in the
prompt. The adapter is thrown away and rebuilt from phase 1 for the next
payload, so no instance contaminates another.

Evidence recall is measured directly before the probes: the model is
cued to reproduce the log and the fraction of observation triples it
returns is recorded. That separates "did not absorb the evidence" from
"absorbed it and still cannot use it", which is the distinction the
August dose sweep was built to make.

## What August found, for comparison

One-world finetuning without a phase-1 adapter failed at every dose: at
15 steps recall was 0, at 25 recall was 0.19 and zero of four replies
were JSON, at 50 and 100 recall was 1.00 and still zero of four were
JSON. The model emitted other prompts instead of answers, because the
evidence string opens with the task header and training on it teaches the
model to produce system prompts.

With the phase-1 adapter underneath, that collapse did not happen: 4 of 4
real JSON replies at 25 steps and recall 0.187 -> 0.547. The answers were
still wrong. So the anchor training prevents the collapse but does not
make weight-delivered evidence usable. This notebook tests that on more
than one seed.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os, shutil
p = f"{OUT_DIR}/raw_phase2_augustopt.jsonl"
if os.path.exists(p):
    os.rename(p, f"{OUT_DIR}/raw_phase2_highdose_325steps.jsonl")
shutil.rmtree(f"{OUT_DIR}/phase2_tmp", ignore_errors=True)
print(sorted(os.listdir(OUT_DIR)))

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, re, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

ADAPTER_MATCH = "augustopt"      # which phase-1 adapter this run builds on
RUN_TAG       = "phase2_augustopt"

REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
_hits = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
_cands = [h for h in _hits if ADAPTER_MATCH in h]
if not _cands and len(_hits) == 1:
    print(f"warning: no path contains {ADAPTER_MATCH!r}, using the only adapter")
    _cands = _hits
if len(_cands) != 1:
    raise SystemExit(f"need exactly one adapter matching {ADAPTER_MATCH!r}, "
                     f"found {_cands or _hits}")
ADAPTER_PATH = os.path.dirname(_cands[0])
PAY_CHANGED  = sorted(glob.glob("/kaggle/input/**/payloads_silent_break_det",
                                recursive=True))[0]
PAY_NOCHANGE = sorted(glob.glob("/kaggle/input/**/payloads_no_change_det",
                                recursive=True))[0]
OUT_DIR = "/kaggle/working"

print("repo:    ", REPO_PATH)
print("eval:    ", EVAL_PATH)
print("adapter: ", ADAPTER_PATH)

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8            # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"gpu: {GPU} | capability {CAP[0]}.{CAP[1]} | compute dtype: {DTYPE}")

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
MODEL_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"
N_SEEDS       = 8         # each seed costs two trainings, changed + no_change
PHASE2_STEPS  = 25        # August's arm D dose
PHASE2_LR     = 2e-4
MAX_LEN       = 2048
RECALL_CUE    = "Recall the observation log for this network."
RECALL_TOKENS = 1600

cfg = json.load(open(os.path.join(ADAPTER_PATH, "adapter_config.json")))
assert cfg["base_model_name_or_path"] == MODEL_NAME, cfg["base_model_name_or_path"]
assert cfg["lora_dropout"] == 0.0, cfg["lora_dropout"]
print("adapter ok:", cfg["base_model_name_or_path"], "| r", cfg["r"])
prov = os.path.join(ADAPTER_PATH, "phase1_provenance.json")
if os.path.isfile(prov):
    p = json.load(open(prov))
    print("phase 1:", p.get("run", "?"), "|", p["anchor_worlds"], "worlds |",
          p["optimizer_steps"], "steps | loss", round(p["final_loss"], 4))

sys.path.insert(0, EVAL_PATH)
import ecpm_eval as E
E.attach(REPO_PATH)

changed  = E.load_payloads(PAY_CHANGED)
nochange = E.load_payloads(PAY_NOCHANGE)
seeds = E.common_seeds(changed, nochange)[:N_SEEDS]
changed  = [p for p in changed  if p["seed"] in seeds]
nochange = [p for p in nochange if p["seed"] in seeds]
payloads = changed + nochange
print(f"\n{len(seeds)} seeds: {seeds}")
print(f"{len(payloads)} payloads -> {len(payloads)} phase-2 trainings")

## Evidence, and the recall metric

The evidence block is the part of the prompt the four probes share. It is
what phase 2 writes into the weights. Recall counts how many of its
observation triples come back when the model is cued for them, which is
the check the dose sweep was built around.

In [ ]:
TRIPLE = re.compile(r"\([A-Z], a\d+, [A-Z]\)")

def evidence_of(pay):
    """The shared context: everything before the probe question."""
    probe = pay["probes"][0]
    full, ask = pay["single"][probe], None
    marker = "Observations, period B:"
    tail = full.split(marker, 1)[1]
    body, ask = tail.split("\n\n", 1) if "\n\n" in tail else (tail, "")
    return full[:full.index(marker)] + marker + body

def recall_of(pay, text):
    want = set(TRIPLE.findall(evidence_of(pay)))
    got = set(TRIPLE.findall(text or ""))
    return len(want & got) / len(want) if want else None

sample = evidence_of(payloads[0])
print(f"evidence: {len(sample)} chars, "
      f"{len(set(TRIPLE.findall(sample)))} distinct triples")
print(sample[:200].replace("\n", " | "), "...")

## Model

The base loads once. For every payload the phase-1 adapter is attached
fresh with `is_trainable=True`, trained, used, then unloaded. Without
`is_trainable=True` the LoRA parameters come back frozen, because
`save_pretrained` writes `inference_mode: true`, and training would
silently do nothing. The assert below catches that.

In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import PeftModel, prepare_model_for_kbit_training
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
print("base loaded | GPU GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

def as_ids(x):
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}")
    return x

def chat_example(user, answer):
    prefix = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": user}],
        add_generation_prompt=True, tokenize=True))
    ans = as_ids(tok(answer + tok.eos_token,
                     add_special_tokens=False)["input_ids"])
    return {"input_ids": prefix + ans,
            "labels": [-100] * len(prefix) + ans}

class OneShot(Dataset):
    def __init__(self, row, n):
        self.rows = [row] * n
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return dict(self.rows[i])

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

def fresh_adapter():
    """Phase-1 weights, trainable, on the shared base."""
    m = PeftModel.from_pretrained(base, ADAPTER_PATH, is_trainable=True)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    assert n > 0, ("no trainable parameters: PeftModel needs "
                   "is_trainable=True, adapter_config has inference_mode true")
    return m

_m = fresh_adapter()
print("trainable params:",
      sum(p.numel() for p in _m.parameters() if p.requires_grad))
base = _m.unload()
print("unloaded cleanly | GPU GB:",
      round(torch.cuda.memory_allocated() / 1e9, 2))

## Run

Per payload: attach phase-1 weights, train `PHASE2_STEPS` on that
payload's evidence, measure recall, probe with an empty context (`ft`)
and with the evidence in the prompt (`combined`), then unload.

Rows stream to JSONL, so a timeout costs only the payload in flight.

In [ ]:
def generate(model, messages, max_new_tokens):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)

rows, recalls = [], {}
rows_path = os.path.join(OUT_DIR, f"raw_{RUN_TAG}.jsonl")
done = set()
if os.path.exists(rows_path):
    for line in open(rows_path):
        r = json.loads(line)
        rows.append(r)
        done.add((r["seed"], r["condition"]))
    print(f"resuming, {len(rows)} rows already saved")

t_start = time.time()
for i, pay in enumerate(payloads, 1):
    key = (pay["seed"], pay["condition"])
    if key in done:
        print(f"[{i}/{len(payloads)}] seed {pay['seed']} {pay['condition']}: done")
        continue
    t0 = time.time()
    model = fresh_adapter()
    model.config.use_cache = False

    row = chat_example(RECALL_CUE, evidence_of(pay))
    assert len(row["input_ids"]) <= MAX_LEN, len(row["input_ids"])
    Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"{OUT_DIR}/phase2_tmp",
            per_device_train_batch_size=1, gradient_accumulation_steps=1,
            max_steps=PHASE2_STEPS, learning_rate=PHASE2_LR,
            warmup_steps=1, lr_scheduler_type="cosine",
            logging_strategy="no", save_strategy="no", report_to=[],
            bf16=USE_BF16, fp16=not USE_BF16,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}),
        train_dataset=OneShot(row, PHASE2_STEPS),
        data_collator=collate).train()

    model.eval()
    model.config.use_cache = True
    text = generate(model, [{"role": "system", "content": E.SYSTEM},
                            {"role": "user", "content": RECALL_CUE}],
                    RECALL_TOKENS)
    rec = recall_of(pay, text)
    recalls[key] = rec

    ask = lambda msgs, n: generate(model, msgs, n)
    new = []
    new += E.run_arm([pay], ask, arm="ft", mode="single",
                     with_evidence=False, verbose=False)
    new += E.run_arm([pay], ask, arm="combined", mode="single",
                     with_evidence=True, verbose=False)
    for r in new:
        r["phase2_steps"] = PHASE2_STEPS
        r["evidence_recall"] = rec
    with open(rows_path, "a") as f:
        for r in new:
            f.write(json.dumps(r) + "\n")
    rows += new

    base = model.unload()
    del model
    torch.cuda.empty_cache()
    print(f"[{i}/{len(payloads)}] seed {pay['seed']} {pay['condition']}: "
          f"recall {rec:.2f}, {time.time()-t0:.0f}s")

print(f"\n{len(rows)} rows in {(time.time()-t_start)/60:.1f} min")
json.dump({f"{k[0]}_{k[1]}": v for k, v in recalls.items()},
          open(os.path.join(OUT_DIR, f"recall_{RUN_TAG}.json"), "w"), indent=1)

## Results

`ft` against `combined` is the question section 6.2 asks. `ft` with an
empty context tests whether absorbed evidence is usable; `combined` tests
whether having it both ways helps or interferes.

Read recall first. An arm that never absorbed the evidence cannot be said
to have failed to use it.

In [ ]:
import pandas as pd

vals = [v for v in recalls.values() if v is not None]
if vals:
    vals.sort()
    print(f"evidence recall over {len(vals)} payloads: min {vals[0]:.2f} "
          f"median {vals[len(vals)//2]:.2f} max {vals[-1]:.2f}")
    print("August: 0.187 before phase 2, 0.547 after, on seed 7")

COLS = ["arm", "sens", "spec", "localize", "pres_acc", "pres_const",
        "target_recall", "pres_parsed", "route_valid", "route_optimal",
        "mean_regret", "bare_json"]
tab = E.table(rows, arms=["ft", "combined"])
display(pd.DataFrame(tab).reindex(columns=COLS))
print("\nroute status")
for r in tab:
    print(" ", r["arm"], r["route_status"])

print("\nMcNemar on localization, combined vs ft:")
print(" ", E.mcnemar([r for r in rows if r["arm"] == "combined"],
                     [r for r in rows if r["arm"] == "ft"]))
json.dump(tab, open(os.path.join(OUT_DIR, f"table_{RUN_TAG}.json"), "w"),
          indent=1)

## Did phase 2 break the format?

The August failure without an anchor adapter was not wrong answers, it
was the model emitting other prompts instead of JSON. `bare_json` and the
parse rate are the check. If they hold up, the anchor training did its
job even where the answers are wrong.

In [ ]:
for arm in ("ft", "combined"):
    a = [r for r in rows if r["arm"] == arm]
    bare = sum(bool(r["bare_json"]) for r in a)
    ok = sum(1 for r in a if r["parsed"].get("status") == "ok")
    print(f"  {arm:9} bare JSON {bare}/{len(a)}   parsed ok {ok}/{len(a)}")
    bad = [r for r in a if r["parsed"].get("status") != "ok"]
    if bad:
        print(f"    first failure ({bad[0]['probe']}): "
              f"{bad[0]['raw'][:110]!r}")

print("\nrecall against localization, per changed payload:")
for r in sorted((r for r in rows if r["probe"] == "localization"
                 and r["scored"].get("applicable")),
                key=lambda r: (r["seed"], r["arm"])):
    p = r["parsed"]
    said = (f"{p.get('node')} {p.get('action')}"
            if p["status"] == "ok" else p["status"])
    print(f"  seed {r['seed']:<3} {r['arm']:9} recall "
          f"{r.get('evidence_recall', float('nan')):.2f}  said {said:12} "
          f"gold {' '.join(r['target'])}  "
          f"{'OK' if r['scored'].get('correct') else ''}")

## Package

In [ ]:
import zipfile, shutil

shutil.rmtree(f"{OUT_DIR}/phase2_tmp", ignore_errors=True)
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path:
            continue
        if ".ipynb_checkpoints" in path or "adapter_model" in path:
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")
for i in zipfile.ZipFile(ZIP_PATH).namelist():
    print("  ", i)